In [ ]:
"""
extract_employees.py
---------------------
Trích xuất bảng EMPLOYEES (đã chuẩn hóa 3NF) từ orders_enriched.csv.

Lý do tách bảng (theo tài liệu ChuanHoaDuLieu.docx):
    orders_enriched vi phạm 3NF vì tồn tại phụ thuộc bắc cầu:
        order_id -> sales_employee_id -> (sales_employee_name,
                                           marital_status,
                                           education_level,
                                           years_experience)
    => Tách các thuộc tính chỉ phụ thuộc vào sales_employee_id
       ra thành bảng Employees riêng, dùng sales_employee_id làm khóa chính.

Đầu ra: silver/Employees.csv
"""

import pandas as pd
from pathlib import Path

RAW_FILE = Path("orders_enriched.csv")
OUT_FILE = Path("silver/Employees.csv")

EMPLOYEE_COLS = [
    "sales_employee_id",
    "sales_employee_name",
    "marital_status",
    "education_level",
    "years_experience",
]


def load_raw(path: Path) -> pd.DataFrame:
    """Đọc trực tiếp các cột cần thiết để tiết kiệm RAM (file gốc ~150MB)."""
    df = pd.read_csv(path, usecols=EMPLOYEE_COLS, encoding="utf-8-sig")
    return df


def clean_and_standardize(df: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hóa dữ liệu: trim khoảng trắng, ép kiểu, xử lý thiếu/trùng."""
    df = df.copy()

    # 1. Chuẩn hóa chuỗi: loại bỏ khoảng trắng thừa
    text_cols = ["sales_employee_id", "sales_employee_name",
                 "marital_status", "education_level"]
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()

    # 2. Chuẩn hóa mã nhân viên về dạng in hoa (EMP0001)
    df["sales_employee_id"] = df["sales_employee_id"].str.upper()

    # 3. Ép kiểu số năm kinh nghiệm
    df["years_experience"] = pd.to_numeric(
        df["years_experience"], errors="coerce"
    ).astype("Int64")

    # 4. Loại bỏ dòng thiếu khóa chính
    before = len(df)
    df = df.dropna(subset=["sales_employee_id"])
    dropped_null_pk = before - len(df)

    # 5. Kiểm tra vi phạm phụ thuộc hàm (1 mã NV phải ứng với đúng 1 bộ thông tin)
    dup_check = df.groupby("sales_employee_id")[
        ["sales_employee_name", "marital_status",
         "education_level", "years_experience"]
    ].nunique()
    violated = dup_check[(dup_check > 1).any(axis=1)]
    if len(violated) > 0:
        print(f"[CẢNH BÁO] {len(violated)} sales_employee_id có dữ liệu "
              f"không nhất quán -> lấy giá trị xuất hiện nhiều nhất (mode).")

    # 6. Rút gọn về 1 dòng / nhân viên (khử trùng lặp -> bảng chiều Employees)
    def pick_mode(s):
        return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

    employees = (
        df.groupby("sales_employee_id", as_index=False)
          .agg({
              "sales_employee_name": pick_mode,
              "marital_status": pick_mode,
              "education_level": pick_mode,
              "years_experience": pick_mode,
          })
          .sort_values("sales_employee_id")
          .reset_index(drop=True)
    )

    print(f"- Số dòng gốc (orders): {before:,}")
    print(f"- Dòng bị loại do thiếu mã NV: {dropped_null_pk}")
    print(f"- Số nhân viên (sau khử trùng lặp): {len(employees):,}")

    return employees


def run():
    print("=== TRÍCH XUẤT BẢNG EMPLOYEES ===")
    raw = load_raw(RAW_FILE)
    employees = clean_and_standardize(raw)

    OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    employees.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
    print(f"-> Đã lưu: {OUT_FILE.resolve()}")
    return employees


if __name__ == "__main__":
    run()

=== TRÍCH XUẤT BẢNG EMPLOYEES ===
- Số dòng gốc (orders): 646,945
- Dòng bị loại do thiếu mã NV: 0
- Số nhân viên (sau khử trùng lặp): 200
-> Đã lưu: /content/silver/Employees.csv


In [ ]:
"""
extract_shippers_shipments.py
------------------------------
Tách shipments_realistic.csv thành 2 bảng đã chuẩn hóa 3NF:

Lý do tách bảng (theo tài liệu ChuanHoaDuLieu.docx):
    order_id -> shipper_id -> toàn bộ thông tin cá nhân/công việc shipper
    (đây là phụ thuộc bắc cầu vi phạm 3NF)

    => SHIPPERS (chiều - dimension): thông tin mô tả người giao hàng,
       khóa chính là shipper_id.
    => SHIPMENTS (sự kiện - fact): mỗi dòng là 1 lượt giao hàng cho 1 đơn,
       khóa ngoại tới Shippers qua shipper_id.

Đầu ra: silver/Shippers.csv, silver/Shipments.csv
"""

import pandas as pd
from pathlib import Path

RAW_FILE = Path("shipments_realistic.csv")
OUT_SHIPPERS = Path("silver/Shippers.csv")
OUT_SHIPMENTS = Path("silver/Shipments.csv")

SHIPPER_ATTR_COLS = [
    "shipper_company", "shipper_vehicle", "shipper_experience_years",
    "shipper_rating", "delivery_success_rate", "average_delivery_time",
    "working_shift", "join_date", "shipper_name", "shipper_phone",
    "shipper_gender", "shipper_age", "shipper_marital_status",
    "shipper_education", "city", "region", "district",
]

SHIPMENT_COLS = ["shipper_id", "order_id", "ship_date",
                  "delivery_date", "shipping_fee"]


def load_raw(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig")


def clean_common(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["shipper_id"] = df["shipper_id"].astype(str).str.strip().str.upper()
    for col in ["ship_date", "delivery_date", "join_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    text_cols = ["shipper_company", "shipper_vehicle", "working_shift",
                 "shipper_name", "shipper_gender", "shipper_marital_status",
                 "shipper_education", "city", "region", "district"]
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
    df["shipping_fee"] = pd.to_numeric(df["shipping_fee"], errors="coerce")
    return df


def build_shippers(df: pd.DataFrame) -> pd.DataFrame:
    """Bảng chiều Shippers - khử trùng lặp theo shipper_id."""
    cols = ["shipper_id"] + SHIPPER_ATTR_COLS
    sub = df[cols].copy()

    def pick_mode(s):
        return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

    dup_check = sub.groupby("shipper_id")[SHIPPER_ATTR_COLS].nunique()
    violated = dup_check[(dup_check > 1).any(axis=1)]
    if len(violated) > 0:
        print(f"[CẢNH BÁO] {len(violated)} shipper_id có dữ liệu không "
              f"nhất quán -> lấy giá trị mode.")

    shippers = (
        sub.groupby("shipper_id", as_index=False)
           .agg({c: pick_mode for c in SHIPPER_ATTR_COLS})
           .sort_values("shipper_id")
           .reset_index(drop=True)
    )

    # Ràng buộc hợp lệ (business rules)
    shippers["shipper_rating"] = shippers["shipper_rating"].clip(0, 5)
    shippers["delivery_success_rate"] = shippers["delivery_success_rate"].clip(0, 100)
    shippers.loc[shippers["shipper_age"] < 18, "shipper_age"] = pd.NA
    shippers.loc[shippers["shipper_experience_years"] < 0, "shipper_experience_years"] = pd.NA

    return shippers


def build_shipments(df: pd.DataFrame) -> pd.DataFrame:
    """Bảng sự kiện Shipments - mỗi dòng 1 lượt giao hàng, PK = order_id."""
    shipments = df[SHIPMENT_COLS].copy()

    before = len(shipments)
    shipments = shipments.drop_duplicates(subset=["order_id"])
    dup_removed = before - len(shipments)

    # Loại các dòng ngày giao trước ngày gửi (dữ liệu phi lý)
    invalid_dates = shipments["delivery_date"] < shipments["ship_date"]
    n_invalid = invalid_dates.sum()
    shipments = shipments[~invalid_dates]

    # Phí ship âm là bất hợp lệ
    shipments = shipments[shipments["shipping_fee"] >= 0]

    print(f"- order_id trùng lặp đã loại: {dup_removed}")
    print(f"- Dòng có delivery_date < ship_date đã loại: {n_invalid}")

    return shipments.sort_values("order_id").reset_index(drop=True)


def run():
    print("=== TRÍCH XUẤT BẢNG SHIPPERS & SHIPMENTS ===")
    raw = load_raw(RAW_FILE)
    print(f"- Số dòng gốc (shipments_realistic): {len(raw):,}")

    cleaned = clean_common(raw)
    shippers = build_shippers(cleaned)
    shipments = build_shipments(cleaned)

    print(f"- Số shipper duy nhất: {len(shippers):,}")
    print(f"- Số lượt giao hàng (Shipments) sau làm sạch: {len(shipments):,}")

    OUT_SHIPPERS.parent.mkdir(parents=True, exist_ok=True)
    shippers.to_csv(OUT_SHIPPERS, index=False, encoding="utf-8-sig")
    shipments.to_csv(OUT_SHIPMENTS, index=False, encoding="utf-8-sig")
    print(f"-> Đã lưu: {OUT_SHIPPERS.resolve()}")
    print(f"-> Đã lưu: {OUT_SHIPMENTS.resolve()}")

    return shippers, shipments


if __name__ == "__main__":
    run()

=== TRÍCH XUẤT BẢNG SHIPPERS & SHIPMENTS ===
- Số dòng gốc (shipments_realistic): 566,067
- order_id trùng lặp đã loại: 0
- Dòng có delivery_date < ship_date đã loại: 0
- Số shipper duy nhất: 80
- Số lượt giao hàng (Shipments) sau làm sạch: 566,067
-> Đã lưu: /content/silver/Shippers.csv
-> Đã lưu: /content/silver/Shipments.csv


In [ ]:
"""
extract_payments.py
---------------------
Trích xuất & chuẩn hóa bảng PAYMENTS từ payments.csv.

Ghi chú chuẩn hóa:
    payments.csv vốn ĐÃ đạt chuẩn 3NF (mỗi order_id ứng với đúng 1 dòng
    thanh toán, các cột payment_method/payment_value/installments chỉ
    phụ thuộc vào order_id, không có phụ thuộc bắc cầu nào) -> không cần
    tách bảng, chỉ cần làm sạch/chuẩn hóa dữ liệu.

    order_id là khóa chính, đồng thời là khóa ngoại tham chiếu tới bảng
    Orders (mỗi đơn hàng có đúng 1 giao dịch thanh toán).

Đầu ra: silver/Payments.csv
"""

import pandas as pd
from pathlib import Path

RAW_FILE = Path("payments.csv")
OUT_FILE = Path("silver/Payments.csv")

PAYMENT_COLS = ["order_id", "payment_method", "payment_value", "installments"]

# Danh sách phương thức thanh toán hợp lệ, dùng để phát hiện giá trị rác
VALID_PAYMENT_METHODS = {
    "credit_card", "paypal", "cod", "apple_pay", "bank_transfer"
}


def load_raw(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, usecols=PAYMENT_COLS, encoding="utf-8-sig")


def clean_and_standardize(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Ép kiểu khóa chính/ngoại
    df["order_id"] = pd.to_numeric(df["order_id"], errors="coerce").astype("Int64")

    # 2. Chuẩn hóa chuỗi phương thức thanh toán: trim + viết thường
    df["payment_method"] = (
        df["payment_method"].astype(str).str.strip().str.lower()
    )

    # 3. Ép kiểu số
    df["payment_value"] = pd.to_numeric(df["payment_value"], errors="coerce")
    df["installments"] = pd.to_numeric(df["installments"], errors="coerce").astype("Int64")

    # 4. Loại dòng thiếu khóa chính
    before = len(df)
    df = df.dropna(subset=["order_id"])
    dropped_null_pk = before - len(df)

    # 5. Loại trùng lặp theo khóa chính order_id (giữ dòng đầu tiên)
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["order_id"], keep="first")
    dropped_dup = before_dedup - len(df)

    # 6. Kiểm tra & gắn cờ giá trị bất hợp lệ (không xóa, chỉ cảnh báo)
    invalid_method = ~df["payment_method"].isin(VALID_PAYMENT_METHODS)
    invalid_value = df["payment_value"] < 0
    invalid_installments = df["installments"] < 1

    if invalid_method.sum() > 0:
        print(f"[CẢNH BÁO] {invalid_method.sum()} dòng có payment_method "
              f"lạ: {df.loc[invalid_method, 'payment_method'].unique()}")
    if invalid_value.sum() > 0:
        print(f"[CẢNH BÁO] {invalid_value.sum()} dòng có payment_value âm")
    if invalid_installments.sum() > 0:
        print(f"[CẢNH BÁO] {invalid_installments.sum()} dòng có installments < 1")

    # 7. Loại các dòng có giá trị thanh toán âm hoặc số kỳ hạn phi lý
    df = df[~invalid_value & ~invalid_installments]

    print(f"- Số dòng gốc: {before:,}")
    print(f"- Dòng bị loại do thiếu order_id: {dropped_null_pk}")
    print(f"- Dòng bị loại do trùng order_id: {dropped_dup}")
    print(f"- Số dòng Payments sau chuẩn hóa: {len(df):,}")

    return df.sort_values("order_id").reset_index(drop=True)


def run():
    print("=== TRÍCH XUẤT BẢNG PAYMENTS ===")
    raw = load_raw(RAW_FILE)
    payments = clean_and_standardize(raw)

    OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    payments.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
    print(f"-> Đã lưu: {OUT_FILE.resolve()}")
    return payments


if __name__ == "__main__":
    run()

=== TRÍCH XUẤT BẢNG PAYMENTS ===
- Số dòng gốc: 646,945
- Dòng bị loại do thiếu order_id: 0
- Dòng bị loại do trùng order_id: 0
- Số dòng Payments sau chuẩn hóa: 646,945
-> Đã lưu: /content/silver/Payments.csv
